# Notebook 12 — Pipeline RAG
Chunking, embeddings, índice FAISS y búsqueda semántica.

In [5]:
import sys, os
sys.path.append(os.path.abspath('..'))
import pandas as pd
from src.Proyecto_3.rag_utils import _get_model, chunk_por_cancion,chunk_por_estrofa, construir_indice,buscar_chunks

df = pd.read_csv('../data//processed/lyrics_clean.csv')
GENEROS = ['Rock', 'Hip-Hop', 'Metal']
df = df[df['Genre'].isin(GENEROS)].reset_index(drop=True)
print(f'Canciones cargadas: {len(df)}')

Canciones cargadas: 3180


## Comparación de estrategias de chunking

In [6]:
# Estrategia 1: canción completa
chunks_cancion = chunk_por_cancion(df)
print(f'Estrategia 1 - Por canción completa: {len(chunks_cancion)} chunks')
print(f'  Longitud promedio: {sum(len(c["texto"]) for c in chunks_cancion)/len(chunks_cancion):.0f} chars')

# Estrategia 2: por estrofa
chunks_estrofa = chunk_por_estrofa(df)
print(f'\nEstrategia 2 - Por estrofa: {len(chunks_estrofa)} chunks')
print(f'  Longitud promedio: {sum(len(c["texto"]) for c in chunks_estrofa)/len(chunks_estrofa):.0f} chars')

Estrategia 1 - Por canción completa: 3180 chunks
  Longitud promedio: 1063 chars

Estrategia 2 - Por estrofa: 3180 chunks
  Longitud promedio: 732 chars


In [7]:
# Análisis comparativo
print('=== COMPARACIÓN DE ESTRATEGIAS ===')
print(f'Canción completa: {len(chunks_cancion)} chunks — más contexto por chunk, menos granularidad')
print(f'Por estrofa:      {len(chunks_estrofa)} chunks — más granularidad, búsqueda más precisa')
print('\nDecisión: usamos estrategia por CANCIÓN COMPLETA para el índice principal,')
print('ya que preserva mejor el contexto temático completo de cada canción.')

=== COMPARACIÓN DE ESTRATEGIAS ===
Canción completa: 3180 chunks — más contexto por chunk, menos granularidad
Por estrofa:      3180 chunks — más granularidad, búsqueda más precisa

Decisión: usamos estrategia por CANCIÓN COMPLETA para el índice principal,
ya que preserva mejor el contexto temático completo de cada canción.


In [8]:
# Construir índice FAISS con la estrategia elegida
# (usa caché si ya existe)
indice, chunks = construir_indice(chunks_cancion)
print(f'\nÍndice FAISS listo con {indice.ntotal} vectores.')

Generando embeddings para 3180 chunks...
Cargando modelo de embeddings...


C:\Users\Roberto\Analisis-Morfosintactico-de-Letras-Musicales\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Roberto\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/1

Embeddings guardados en caché.
Índice FAISS construido con 3180 vectores.

Índice FAISS listo con 3180 vectores.


In [9]:
# Pruebas de búsqueda semántica
queries = [
    'songs about rebellion and freedom',
    'dark heavy aggressive lyrics',
    'street life urban poetry',
]

for q in queries:
    print(f'\n🔍 Query: "{q}"')
    resultados = buscar_chunks(q, top_k=3)
    for r in resultados:
        print(f'  [{r["genre"]}] {r["song"]} — {r["artist"]} (score: {r["score"]:.3f})')


🔍 Query: "songs about rebellion and freedom"
  [Hip-Hop] the-grits — cappadonna (score: 0.644)
  [Hip-Hop] what-s-on-my-mind-ii — dayton-family (score: 0.578)
  [Metal] tempo-of-the-damned — exodus (score: 0.528)

🔍 Query: "dark heavy aggressive lyrics"
  [Hip-Hop] smithzonian-institute-of-rhyme — blackalicious (score: 0.551)
  [Metal] by-dark-glorious-thoughts — enthroned (score: 0.548)
  [Metal] screams-go-unheard — cryptopsy (score: 0.537)

🔍 Query: "street life urban poetry"
  [Rock] eight-miles-high — emerson-lake-palmer (score: 0.584)
  [Metal] a-new-sun-rises — anam-cara (score: 0.548)
  [Rock] road-signs-rock-songs — ataris (score: 0.545)


In [10]:
# Prueba con filtro de género
print('🔍 Búsqueda filtrada por género Metal:')
resultados = buscar_chunks('anger and power', top_k=3, filtro_genero='Metal')
for r in resultados:
    print(f'  {r["song"]} — {r["artist"]} (score: {r["score"]:.3f})')

🔍 Búsqueda filtrada por género Metal:
  damage-path — diecast (score: 0.627)
  rage-hard — atrocity (score: 0.615)
  backlash — front-line-assembly (score: 0.614)
